# M2 — Agent Layer Demo

End-of-milestone demonstration: run Agent 1 (news sentiment) for a single rebalance date and inspect its output.

## Prerequisites
```bash
uv run python scripts/init_db.py
uv run python scripts/ingest_alpha_vantage_news.py  # historical backfill (or ingest_news.py for recent)
```

## Sections
1. **Run NewsAgent** — prepare input, call LLM (cached), validate output
2. **Sentiment bar chart** — per-sector sentiment scores for the analysis week
3. **Signals table** — rows written to the `signals` table in SQLite

In [ ]:
import datetime
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

DB_PATH = Path('..') / 'data' / 'state.db'
engine = create_engine(f'sqlite:///{DB_PATH}')

# Analysis date — must be within the news_raw date range.
# Change this to any Monday that has news in the DB.
ANALYSIS_DATE = datetime.date(2024, 6, 7)

SECTOR_NAMES = {
    'XLK': 'Technology',          'XLF': 'Financials',
    'XLV': 'Health Care',         'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',    'XLE': 'Energy',
    'XLI': 'Industrials',         'XLB': 'Materials',
    'XLRE': 'Real Estate',        'XLU': 'Utilities',
}

print(f'Analysis date : {ANALYSIS_DATE}')
print(f'News window   : {ANALYSIS_DATE - datetime.timedelta(days=7)}  →  {ANALYSIS_DATE}')
print(f'DB path       : {DB_PATH.resolve()}')

## Section 1 — Run NewsAgent

The agent queries `news_raw` for the trailing 7 days, formats a structured prompt,
calls `claude-haiku-4-5-20251001` (or returns cached response on repeat runs),
validates the JSON output, and writes rows to the `signals` table.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

from agents.news_agent import NewsAgent

agent = NewsAgent()

# Check article coverage before calling the LLM
input_data = agent.prepare_input(ANALYSIS_DATE, engine)
coverage = {etf: len(articles) for etf, articles in input_data['sectors'].items()}
total_articles = sum(coverage.values())

print(f'\nArticle coverage for week {input_data["week_start"]} → {input_data["analysis_date"]}:')
for etf, n in sorted(coverage.items()):
    bar = '█' * min(n, 20)
    print(f'  {etf:5s}  {n:3d}  {bar}')
print(f'\nTotal: {total_articles} articles across {len(coverage)} sectors')

if total_articles == 0:
    print('\n⚠  No news data for this date range — run the ingestion scripts first.')
    print('   Try a more recent date if backfill is still in progress.')

In [ ]:
if total_articles == 0:
    print('⚠  Skipping LLM call — no articles in window.')
    result = None
else:
    print('Calling NewsAgent.run() ...')
    result = agent.run(ANALYSIS_DATE, engine)
    print('\n✓ Agent completed successfully')
    print(f'  conviction  : {result["conviction"]:.2f}')
    print(f'  key_themes  : {result["key_themes"]}')
    print(f'\nSector sentiments:')
    for etf, score in sorted(result['sector_sentiments'].items()):
        direction = '▲' if score > 0.05 else ('▼' if score < -0.05 else '─')
        print(f'  {etf:5s}  {score:+.2f}  {direction}')

## Section 2 — Sentiment bar chart

Horizontal bar chart of per-sector sentiment scores.  
Green = positive, red = negative, grey = near-zero (|score| < 0.05).

In [ ]:
if result is None:
    print('⚠  No result to plot.')
else:
    sentiments = result['sector_sentiments']
    labels = [f'{etf} — {SECTOR_NAMES.get(etf, etf)}' for etf in sentiments]
    scores = list(sentiments.values())
    colors = ['#2ecc71' if s > 0.05 else '#e74c3c' if s < -0.05 else '#bdc3c7' for s in scores]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(labels, scores, color=colors, alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-1.05, 1.05)
    ax.set_xlabel('Sentiment score  (−1 = strong bearish, +1 = strong bullish)', fontsize=10)
    ax.set_title(
        f'News Sentiment by Sector\n'
        f'Week {input_data["week_start"]} → {input_data["analysis_date"]}  '
        f'| conviction = {result["conviction"]:.2f}',
        fontsize=12,
    )
    for bar, score in zip(bars, scores):
        offset = 0.02 if score >= 0 else -0.02
        ha = 'left' if score >= 0 else 'right'
        ax.text(score + offset, bar.get_y() + bar.get_height() / 2,
                f'{score:+.2f}', va='center', ha=ha, fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print('Key themes:')
    for i, theme in enumerate(result['key_themes'], 1):
        print(f'  {i}. {theme}')

## Section 3 — Signals table

Rows written to `signals` for `agent_name = 'sentiment'` on the analysis date.

In [ ]:
sig_df = pd.read_sql(
    text("""
        SELECT s.date, s.target, s.signal_value, s.confidence, s.raw_call_id,
               a.model_string, a.cached, a.cost_usd, a.latency_ms
        FROM signals s
        LEFT JOIN agent_calls a ON s.raw_call_id = a.call_id
        WHERE s.agent_name = 'sentiment'
        ORDER BY s.date DESC, s.target
    """),
    engine,
)

if sig_df.empty:
    print('⚠  No sentiment signals in DB yet — run the agent cell above first.')
else:
    sig_df['signal_value'] = sig_df['signal_value'].round(3)
    sig_df['cost_usd'] = sig_df['cost_usd'].map(lambda x: f'${x:.5f}' if x is not None else '—')
    sig_df['latency_ms'] = sig_df['latency_ms'].map(lambda x: f'{x:.0f}ms' if x is not None else '—')
    sig_df['cached'] = sig_df['cached'].map(lambda x: '✓' if x else '✗')
    display(sig_df.rename(columns={
        'target': 'Sector', 'signal_value': 'Sentiment', 'confidence': 'Conviction',
        'model_string': 'Model', 'cost_usd': 'Cost', 'latency_ms': 'Latency', 'cached': 'Cached',
    }))
    print(f'\nTotal signal rows in DB: {len(sig_df)}')